# Decision intelligence

Monitoring and investigation priorities for observed England atorvastatin activity, not market opportunity or clinical advice.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root))
from src.decision_features import configuration
from src.prioritization import priority_scores
from src.scenario_engine import accounting, break_even, mix_simulation
cfg=configuration(root)
def table(name): return pd.read_parquet(root/f'outputs/phase5/tables/{name}.parquet')
def chart(name): display(Image(filename=str(root/f'outputs/phase5/charts/{name}.png')))


## Features and observed history

Grains are unique. Observed rows union absence-only cells preserves the calendar; absent activity is unknown, not zero.

In [ ]:
display(table('executive_kpi_summary'))
display(table('regional_decision_features'))
display(table('presentation_change_contributions'))

## Transparent policy priorities

Five retained components: activity 25%, absolute change 20%, cost-pattern change 20%, forecast error 20%, data quality 15%. Percentile scores use midranks; missing required dimensions leave an entity unscored. Scores describe analytical attention, not commercial value.

In [ ]:
f=table('regional_decision_features')
recalculated=priority_scores(f,cfg)
assert recalculated.entity_id.tolist()==table('regional_priority_scores').entity_id.tolist()
display(table('regional_priority_scores')[['region_name','activity_score','change_score','cost_score','forecast_score','quality_score','composite_score','rank','monitoring_tier']])
display(table('presentation_priority_scores')[['presentation_name','rank','monitoring_tier','missing_components']])
chart('01_regional_priority_matrix')

## Sensitivity and rank stability

Equal/activity/forecast/data-quality weights, leave-one-component-out, min-max normalization and three tier settings expose policy dependence. A changed rank is not probabilistic uncertainty.

In [ ]:
display(table('rank_stability_summary'))
display(table('priority_rank_instability').query("aggregation_level == 'regional'"))
chart('04_priority_sensitivity')
chart('05_rank_stability')

## Frozen forecast reliability

Phase 5 does not fit or reselect models. Historical errors are scaled and compared against naive on paired support. Six months of overlapping holdout targets cannot establish future guarantees; no calibrated intervals are available.

In [ ]:
display(table('forecast_reliability_profiles').query("aggregation_level != 'region_presentation'"))
display(table('forecast_horizon_risk').query("series_id == 'England' and split_type == 'holdout'"))
chart('09_regional_reliability')
chart('12_hierarchy_gaps')

## Rule-based analyst investigations

Each proposed action records its trigger, source key, observed value, evidence class and limitation. No clinical, manufacturer-price or resource-investment recommendation is generated.

In [ ]:
display(table('executive_recommendations').head(20))
display(table('decision_evidence_register'))
assert table('recommendations_dashboard').recommendation_id.is_unique
chart('18_recommendation_priorities')